# CORDIC on the Red Pitaya — graphical explorer

Runs the FPGA CORDIC cores (`cordic_top` and `cordic_sqrt_wrapper`) on this board and plots the hardware results against exact math.

* **Circular CORDIC:** a sine/cosine sweep, the unit circle, rotation of an arbitrary vector, and the error in LSB.
* **Square root:** √x over the whole 32-bit range, with the error in Q16.16 LSB.
* **Performance:** latency and throughput measured with the on-chip cycle counters.
* **Interactive:** move the sliders to rotate a vector on the FPGA.

Every value plotted as "FPGA" was computed by the hardware through the AXI registers at `0x4000_0000`.

> **Run the first code cell before anything else.** It loads the CORDIC bitstream into the FPGA. If you open a Red Pitaya web app or another notebook that reloads the FPGA, run the first cell again before using the other cells. Accessing the registers while another bitstream is loaded freezes the board, and it then needs a power cycle.


In [ ]:
import os, sys, subprocess, time
import numpy as np
import matplotlib.pyplot as plt

HERE = os.path.abspath(".")
sys.path.insert(0, HERE)
import cordic_hw_test as hw

PLOTS = os.path.join(HERE, "plots")
os.makedirs(PLOTS, exist_ok=True)

# Always (re)load the CORDIC bitstream BEFORE touching the AXI bus.
# Never read 0x4000_0000 to "check" what is loaded: if another bitstream is in
# the FPGA (web apps and other notebooks reload it), an unanswered GP0 access
# freezes the whole board. Loading takes ~60 ms.
hw.load_bitstream(os.path.join(HERE, "cordic_rp.bit.bin"))
regs = hw.Regs()
assert regs.rd(hw.ID) == hw.EXPECTED_ID, "CORDIC bitstream not responding"
fclk = hw.measure_fclk(regs)
print(f"Bitstream OK — FCLK0 = {fclk/1e6:.2f} MHz, K = {hw.K:.6f}, 1/K = {hw.INV_K}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Sine and cosine sweep

Input `x0 = 1/K`, `y0 = 0`, with the angle swept over the full circle in 0.5° steps. The FPGA output is cos/sin directly, in Q1.15.

In [ ]:
deg = np.arange(-180, 180, 0.5)
hw_xy = np.array([hw.rotate(regs, hw.INV_K, 0, hw.deg_to_phase(d))[:2] for d in deg])
ref_xy = np.array([hw.expected_rotation(hw.INV_K, 0, hw.deg_to_phase(d)) for d in deg])
err = hw_xy - ref_xy

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1.3]})
ax1.plot(deg, np.cos(np.radians(deg)), color="0.75", lw=4, label="math cos")
ax1.plot(deg, np.sin(np.radians(deg)), color="0.85", lw=4, label="math sin")
ax1.plot(deg, hw_xy[:, 0] / 32768, ".", ms=3, color="C0", label="FPGA x (cos)")
ax1.plot(deg, hw_xy[:, 1] / 32768, ".", ms=3, color="C3", label="FPGA y (sin)")
ax1.set_ylabel("value"); ax1.legend(loc="lower left", ncol=2, fontsize=8)
ax1.set_title(f"Circular CORDIC on the FPGA — {len(deg)} rotations")
ax2.plot(deg, err[:, 0], ".", ms=3, color="C0", label="x error")
ax2.plot(deg, err[:, 1], ".", ms=3, color="C3", label="y error")
ax2.set_ylabel("error [LSB]"); ax2.set_xlabel("angle [deg]")
ax2.set_ylim(-3, 3); ax2.set_xticks(range(-180, 181, 45)); ax2.legend(fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(f"{PLOTS}/1_sin_cos.png"); plt.show()
print(f"max |error|: x = {abs(err[:,0]).max()} LSB, y = {abs(err[:,1]).max()} LSB")

## 2. Unit circle and arbitrary vector rotation

The sweep above plotted in the plane, plus a vector (0.3, 0.4) rotated by the FPGA in 10° steps. Its length grows by the CORDIC gain K, from 0.5 to 0.5·K.

In [ ]:
v0 = (round(0.3 * 32768), round(0.4 * 32768))
angs = np.arange(0, 360, 10)
rot = np.array([hw.rotate(regs, v0[0], v0[1], hw.deg_to_phase(a))[:2] for a in angs]) / 32768

t = np.linspace(0, 2*np.pi, 400)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(np.cos(t), np.sin(t), color="0.8", lw=3, label="unit circle")
ax.plot(hw_xy[::6, 0] / 32768, hw_xy[::6, 1] / 32768, ".", color="C0", ms=5, label="FPGA (1/K, 0) sweep")
r = 0.5 * hw.K
ax.plot(r*np.cos(t), r*np.sin(t), "--", color="0.6", lw=1, label=f"|v|·K = {r:.3f}")
ax.quiver(0, 0, 0.3, 0.4, angles="xy", scale_units="xy", scale=1, color="k", width=0.006, label="input (0.3, 0.4)")
ax.plot(rot[:, 0], rot[:, 1], "o", color="C2", ms=5, label="FPGA rotations of (0.3, 0.4)")
ax.set_aspect("equal"); ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15)
ax.legend(loc="lower left", fontsize=8); ax.set_title("Rotations computed by the FPGA")
fig.tight_layout(); fig.savefig(f"{PLOTS}/2_circle.png"); plt.show()

## 3. Square root over the 32-bit range

The radicand is log-spaced from 1 to 2³²−1, plus 5000 random values for the error histogram. The output is Q16.16.

In [ ]:
rads = np.unique(np.round(np.logspace(0, np.log10(2**32 - 1), 600)).astype(np.uint64))
roots = np.array([hw.sqrt_hw(regs, int(r)) for r in rads], dtype=np.float64)
exact = np.array([hw.expected_root(int(r)) for r in rads], dtype=np.float64)

rng = np.random.default_rng(1)
rand = rng.integers(0, 2**32, 5000, dtype=np.uint64)
rand_err = np.array([hw.sqrt_hw(regs, int(r)) - hw.expected_root(int(r)) for r in rand])

fig, axs = plt.subplots(1, 3, figsize=(13, 4))
axs[0].loglog(rads, np.sqrt(rads.astype(float)), color="0.75", lw=4, label="math √x")
axs[0].loglog(rads, roots / 65536, ".", ms=3, color="C1", label="FPGA")
axs[0].set_xlabel("x"); axs[0].set_ylabel("√x"); axs[0].legend(); axs[0].set_title("√x on the FPGA")
axs[1].semilogx(rads, roots - exact, ".", ms=3, color="C1")
axs[1].set_ylim(-2, 2); axs[1].set_xlabel("x"); axs[1].set_ylabel("error [Q16.16 LSB]")
axs[1].set_title("error vs. exact (log-spaced)")
vals, counts = np.unique(rand_err, return_counts=True)
axs[2].bar(vals, counts, color="C1", width=0.6)
axs[2].set_xticks(range(-2, 3)); axs[2].set_xlabel("error [LSB]"); axs[2].set_ylabel("count")
axs[2].set_title("5000 random inputs")
fig.tight_layout(); fig.savefig(f"{PLOTS}/3_sqrt.png"); plt.show()
print(f"max |error| = {int(max(abs(roots - exact).max(), abs(rand_err).max()))} LSB  (1 LSB = 2^-16 = 1.5e-5)")

## 4. Latency and throughput (on-chip cycle counters)

In [ ]:
# Single-operation latency (start → result), from C_CYCLES / S_CYCLES
hw.rotate(regs, hw.INV_K, 0, 0); lat_c = regs.rd(hw.C_CYCLES)
hw.sqrt_hw(regs, 123456789);      lat_s = regs.rd(hw.S_CYCLES)
cyc_c, _ = hw.burst(regs, hw.C_CTRL, hw.C_STATUS, hw.C_BURST_N, hw.C_CYCLES, hw.C_COUNT, 10000)
cyc_s, _ = hw.burst(regs, hw.S_CTRL, hw.S_STATUS, hw.S_BURST_N, hw.S_CYCLES, hw.S_COUNT, 10000)
per = np.array([cyc_c, cyc_s]) / 10000
names = ["circular", "sqrt"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
b = a1.bar(names, [lat_c, lat_s], color=["C0", "C1"]); a1.bar_label(b, fmt="%d cycles")
a1.set_ylabel("clock cycles"); a1.set_title("latency of one operation")
b = a2.bar(names, fclk / per / 1e6, color=["C0", "C1"]); a2.bar_label(b, fmt="%.2f M/s")
a2.set_ylabel("results / s [M]"); a2.set_title(f"sustained throughput @ {fclk/1e6:.0f} MHz")
for a in (a1, a2): a.margins(y=0.2)
fig.tight_layout(); fig.savefig(f"{PLOTS}/4_performance.png"); plt.show()
print(f"circular: {per[0]:.2f} cycles/result, sqrt: {per[1]:.2f} cycles/result")

## 5. Interactive rotation

Move the sliders: every change sends the vector and angle to the FPGA and draws the hardware result (green) on top of the exact rotation (dashed). Runs only in a live JupyterLab session.

In [ ]:
import ipywidgets as w

def show(angle=30.0, x0=0.5, y0=0.0):
    xi, yi = round(x0 * 32768), round(y0 * 32768)
    xo, yo, zr = hw.rotate(regs, xi, yi, hw.deg_to_phase(angle))
    xe, ye = hw.expected_rotation(xi, yi, hw.deg_to_phase(angle))
    fig, ax = plt.subplots(figsize=(5, 5))
    t = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), color="0.85")
    ax.quiver(0, 0, x0, y0, angles="xy", scale_units="xy", scale=1, color="k", width=0.008)
    ax.quiver(0, 0, xo/32768, yo/32768, angles="xy", scale_units="xy", scale=1, color="C2", width=0.008)
    ax.plot([0, xe/32768], [0, ye/32768], "--", color="C3")
    ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_title(f"FPGA: ({xo}, {yo})   exact: ({xe}, {ye})   residual phase {zr}")
    plt.show()

w.interact(show,
           angle=w.FloatSlider(value=30, min=-180, max=179.9, step=0.5, description="angle [°]"),
           x0=w.FloatSlider(value=0.5, min=-0.6, max=0.6, step=0.01, description="x0"),
           y0=w.FloatSlider(value=0.0, min=-0.6, max=0.6, step=0.01, description="y0"));